# Employees Plus

The following are tasks performed by this notebook
- Reads the fictional employee roster generated in [roster_generator.ipynb](https://github.com/AtypicalLogic/python_learning/blob/primary/mini_projects/hr_analytics/roster_generator.ipynb)
- Performs a Cartesian join between the employee roster and a dates table
- Calculates the following fields
    - `employee_status`
    - `employee_tenure_duration_years`
    - `employee_tenure_duration_buckets`

In [1]:
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

## Globals

In [2]:
start_date = date(2021, 1, 1)
end_date = date(2026, 12, 31)

## Generating `df_date` dataframe

In [3]:
# generating the date table
df_date = pd.DataFrame({'full_date': pd.date_range(start=start_date, end=end_date, freq='D')})

# calculating the date attributes
df_date['date_id'] = df_date['full_date'].dt.strftime('%Y-%m-%d')
df_date['year'] = df_date['full_date'].dt.year
df_date['month'] = df_date['full_date'].dt.month
df_date['day'] = df_date['full_date'].dt.day
df_date['half_name'] = np.where(df_date['month'] < 7, 'H1', 'H2')
df_date['quarter_name'] = 'Q' + df_date['full_date'].dt.quarter.astype(str)
df_date['month_name'] = df_date['full_date'].dt.strftime('%B')
df_date['day_name'] = df_date['full_date'].dt.strftime('%A')

# calculating boolean flags
df_date['is_weekday'] = ~df_date['day_name'].isin(['Saturday', 'Sunday'])
df_date['is_first_day_of_year'] = (df_date['month'] == 1) & (df_date['day'] == 1)
df_date['is_first_day_of_half'] = df_date['is_first_day_of_year'] | ((df_date['month'] == 7) & (df_date['day'] == 1))
df_date['is_first_day_of_quarter'] = df_date['is_first_day_of_year'] | df_date['is_first_day_of_half'] | (df_date['day_name'].isin(['April', 'October']) & (df_date['day'] == 1))
df_date['is_first_day_of_month'] = df_date['day'] == 1
df_date['is_last_day_of_year'] = (df_date['month'] == 12) & (df_date['full_date'].dt.is_month_end)
df_date['is_last_day_of_half'] = df_date['is_last_day_of_year'] | ((df_date['month'] == 6) & (df_date['full_date'].dt.is_month_end))
df_date['is_last_day_of_quarter'] = df_date['is_last_day_of_year'] | df_date['is_last_day_of_half'] | (df_date['month'].isin([3, 9]) & (df_date['full_date'].dt.is_month_end))
df_date['is_last_day_of_month'] = df_date['full_date'].dt.is_month_end

# converting `full_date` to date type
df_date['full_date'] = df_date['full_date'].dt.normalize()

In [4]:
df_date.shape

(2191, 18)

## Loading employee roster

In [5]:
df_employees = pd.read_csv('people_report-build - employee.csv')

# converting date fields to datetime objects
df_employees['hire_date'] = pd.to_datetime(df_employees['hire_date']).dt.normalize()
df_employees['termination_date'] = pd.to_datetime(df_employees['termination_date']).dt.normalize()

# filling NaT with the maximum date for comparison, similar to COALESCE in SQL
MAX_DATE = pd.to_datetime('2260-01-01').normalize()
df_employees['termination_date_coalesced'] = df_employees['termination_date'].fillna(MAX_DATE)


## Performing Cartisian join

In [6]:
df_employees_plus = df_employees.merge(
    right=df_date[df_date.is_last_day_of_month][['full_date', 'is_last_day_of_month', 'quarter_name', 'is_first_day_of_quarter', 'is_last_day_of_quarter']], 
    how='cross'
)

In [7]:
print(df_employees_plus.shape)
display(df_employees_plus.head())

(412200, 29)


,employee_id,hire_date,termination_date,org_l01,gender,ethnicity,job_level,ethnicity_is_underrepresented_minority,org_l00,org_l02,...,employee_tenure_current_month_year,employee_tenure_duration_bucket,gender_remapped,ethnicity_remapped,termination_date_coalesced,full_date,is_last_day_of_month,quarter_name,is_first_day_of_quarter,is_last_day_of_quarter
0,e000001,2014-01-02,NaT,NaN,Male,White,M11,False,Company Inc.,NaN,...,11.00069,05--4+ years,Male,00--White,2260-01-01,2021-01-31,True,Q1,False,False
1,e000001,2014-01-02,NaT,NaN,Male,White,M11,False,Company Inc.,NaN,...,11.00069,05--4+ years,Male,00--White,2260-01-01,2021-02-28,True,Q1,False,False
2,e000001,2014-01-02,NaT,NaN,Male,White,M11,False,Company Inc.,NaN,...,11.00069,05--4+ years,Male,00--White,2260-01-01,2021-03-31,True,Q1,False,True
3,e000001,2014-01-02,NaT,NaN,Male,White,M11,False,Company Inc.,NaN,...,11.00069,05--4+ years,Male,00--White,2260-01-01,2021-04-30,True,Q2,False,False
4,e000001,2014-01-02,NaT,NaN,Male,White,M11,False,Company Inc.,NaN,...,11.00069,05--4+ years,Male,00--White,2260-01-01,2021-05-31,True,Q2,False,False


## Calculating derived fields

### `prehire_status`

In [8]:
condition_prehire = (df_employees_plus['full_date'] <= df_employees_plus['hire_date'])

df_employees_plus['prehire_status'] = np.where(
    condition_prehire,
    'Prehire',
    'Not prehire'
)

### `employee_status`

In [9]:
# Check if full_date is between hire_date and the coalesced termination date (inclusive)
condition_active = (df_employees_plus['full_date'] >= df_employees_plus['hire_date']) & \
                   (df_employees_plus['full_date'] <= df_employees_plus['termination_date_coalesced'])


df_employees_plus['employee_status'] = np.where(
    condition_active,
    'Active',
    'Terminated'
)

df_employees_plus.loc[df_employees_plus['prehire_status'] == 'Prehire', 'employee_status'] = 'n/a'

### `employee_tenure_duration_years`

In [10]:
DAYS_IN_YEAR = 365.25

# identifying appropriate end date for tenure
# if active and not prehire, tenure ends at full_date; if terminated, tenure ends at actual termination_date
tenure_end_date = np.where(
    (df_employees_plus['employee_status'] == 'Active') & (df_employees_plus['prehire_status'] == 'Not prehire'),
    df_employees_plus['full_date'],
    df_employees_plus['termination_date']
)
tenure_end_date = pd.Series(tenure_end_date).dt.normalize()

# calculating tenure in days
tenure_days = (tenure_end_date - df_employees_plus['hire_date']).dt.days

# calculating tenure in years
df_employees_plus['employee_tenure_duration_years'] = (tenure_days / DAYS_IN_YEAR).round(3)

# Handle cases where an employee is active *before* their hire date due to the nature of the cross join (set to 0)
df_employees_plus.loc[df_employees_plus['employee_tenure_duration_years'] < 0, 'employee_tenure_duration_years'] = 0
df_employees_plus.loc[df_employees_plus.full_date <= df_employees_plus.hire_date, 'employee_tenure_duration_years'] = 0

### `employee_tenure_duration_bucket`

In [11]:
# Define the bins (boundaries) and labels
bins = [0, 0.5, 1, 2, 3, 4, np.inf]
labels = ['0-0.5 years', '0.5-1 years', '1-2 years', '2-3 years', '3-4 years', '4+ years']

# Use pd.cut to segment the data
df_employees_plus['employee_tenure_duration_bucket'] = pd.cut(
    df_employees_plus['employee_tenure_duration_years'],
    bins=bins,
    labels=labels,
    right=False, # makes the intervals [lower, upper)
    include_lowest=True
).astype(str)

## Validations

Validating employee 'e004943' for status on '2023-11-30'

In [12]:
df_employees_plus[
    (df_employees_plus.full_date == '2023-11-30')
    & (df_employees_plus.hire_date > '2023-11-30')
    & (df_employees_plus.employee_id == 'e004943')
][['full_date', 'employee_id', 'hire_date', 'termination_date', 'prehire_status', 'employee_status', 'employee_tenure_duration_years', 'employee_tenure_duration_bucket']]


,full_date,employee_id,hire_date,termination_date,prehire_status,employee_status,employee_tenure_duration_years,employee_tenure_duration_bucket
355858,2023-11-30,e004943,2023-12-25,2024-05-28,Prehire,n/a,0.0,0-0.5 years


Validating prehires on '2023-11-30'

In [13]:
df_employees_plus[
    (df_employees_plus.full_date == '2023-11-30')
    & (df_employees_plus.hire_date > '2023-11-30')
][['full_date', 'employee_id', 'hire_date', 'termination_date', 'prehire_status', 'employee_status', 'employee_tenure_duration_years', 'employee_tenure_duration_bucket']]


,full_date,employee_id,hire_date,termination_date,prehire_status,employee_status,employee_tenure_duration_years,employee_tenure_duration_bucket
355714,2023-11-30,e004941,2023-12-19,NaT,Prehire,n/a,0.0,0-0.5 years
355786,2023-11-30,e004942,2023-12-14,NaT,Prehire,n/a,0.0,0-0.5 years
355858,2023-11-30,e004943,2023-12-25,2024-05-28,Prehire,n/a,0.0,0-0.5 years
355930,2023-11-30,e004944,2023-12-15,NaT,Prehire,n/a,0.0,0-0.5 years
356002,2023-11-30,e004945,2023-12-29,NaT,Prehire,n/a,0.0,0-0.5 years
...,...,...,...,...,...,...,...,...
411874,2023-11-30,e005721,2024-11-28,NaT,Prehire,n/a,0.0,0-0.5 years
411946,2023-11-30,e005722,2024-11-25,NaT,Prehire,n/a,0.0,0-0.5 years
412018,2023-11-30,e005723,2024-11-21,NaT,Prehire,n/a,0.0,0-0.5 years
412090,2023-11-30,e005724,2024-11-12,NaT,Prehire,n/a,0.0,0-0.5 years


Validating no prehires with tenure > 0

In [14]:
df_employees_plus[
    (df_employees_plus.prehire_status == 'Prehire')
    & (df_employees_plus.employee_tenure_duration_years > 0)
][['full_date', 'employee_id', 'hire_date', 'termination_date', 'prehire_status', 'employee_status', 'employee_tenure_duration_years', 'employee_tenure_duration_bucket']]


,full_date,employee_id,hire_date,termination_date,prehire_status,employee_status,employee_tenure_duration_years,employee_tenure_duration_bucket


Validating employee 'e004943' for status progression

In [15]:
df_employees_plus[
    (df_employees_plus.employee_id == 'e004943')
    & (df_employees_plus.full_date.isin(['2021-01-31', '2023-11-30', '2023-12-31', '2024-01-31', '2024-04-30', '2024-05-31', '2024-12-31']))
][['full_date', 'employee_id', 'hire_date', 'termination_date', 'prehire_status', 'employee_status', 'employee_tenure_duration_years', 'employee_tenure_duration_bucket']]


,full_date,employee_id,hire_date,termination_date,prehire_status,employee_status,employee_tenure_duration_years,employee_tenure_duration_bucket
355824,2021-01-31,e004943,2023-12-25,2024-05-28,Prehire,n/a,0.000,0-0.5 years
355858,2023-11-30,e004943,2023-12-25,2024-05-28,Prehire,n/a,0.000,0-0.5 years
355859,2023-12-31,e004943,2023-12-25,2024-05-28,Not prehire,Active,0.016,0-0.5 years
355860,2024-01-31,e004943,2023-12-25,2024-05-28,Not prehire,Active,0.101,0-0.5 years
355863,2024-04-30,e004943,2023-12-25,2024-05-28,Not prehire,Active,0.348,0-0.5 years
355864,2024-05-31,e004943,2023-12-25,2024-05-28,Not prehire,Terminated,0.424,0-0.5 years
355871,2024-12-31,e004943,2023-12-25,2024-05-28,Not prehire,Terminated,0.424,0-0.5 years


Validating employee 'e005725' for status progression

In [16]:
df_employees_plus[
    (df_employees_plus.employee_id == 'e005725')
    & (df_employees_plus.full_date.isin(['2021-01-31', '2024-10-31', '2024-11-30', '2024-12-31', '2025-12-31']))
][['full_date', 'employee_id', 'hire_date', 'termination_date', 'prehire_status', 'employee_status', 'employee_tenure_duration_years', 'employee_tenure_duration_bucket']]


,full_date,employee_id,hire_date,termination_date,prehire_status,employee_status,employee_tenure_duration_years,employee_tenure_duration_bucket
412128,2021-01-31,e005725,2024-11-25,NaT,Prehire,n/a,0.000,0-0.5 years
412173,2024-10-31,e005725,2024-11-25,NaT,Prehire,n/a,0.000,0-0.5 years
412174,2024-11-30,e005725,2024-11-25,NaT,Not prehire,Active,0.014,0-0.5 years
412175,2024-12-31,e005725,2024-11-25,NaT,Not prehire,Active,0.099,0-0.5 years
412187,2025-12-31,e005725,2024-11-25,NaT,Not prehire,Active,1.098,1-2 years


In [17]:
df_employees_plus.to_csv('employee_data_plus.csv')